<h1 style="font-size: 36px; color: blue;">STOWA Proevenverzameling tool v5.0</h1>

Deze Jupyter notebook bevat functies opgenomen in de python package PV-tooling ### voor het opstellen van lokale of regionale proevenverzamelingen voor het bepalen van geotechnische parameters. De methode is ontwikkeld voor het uitvoeren van analyses in relatie tot de geotechnische stabiliteit van dijken, maar kan ook breder worden toegepast. De notebook dient tevens als handleiding om de gebruiker stapsgewijs te ondersteunen bij het opstellen van een proevenverzameling.

Met de beschikbare functies kunnen zowel gedraineerde als ongedraineerde sterkteparameters worden berekend alsmede enkele samendrukkingsparameters. Van deze parameters worden verwachtingswaarde, karakteristieke waarde en rekenwaarde bepaald.

De functies zijn opgesteld conform de werkwijze beschreven in [Statistische methoden t.b.v. proevenverzamelingen, DIV, v1.0], zie tevens: https://publicwiki.deltares.nl/spaces/HWBPMacro/pages/217120830/Sterkte+van+grond#Sterktevangrond-150. De tool bevat tevens hulpmiddelen voor het onderscheiden of samenvoegen van groepen in een verzameling op basis van verschillende kenmerken.

De onderliggende data om een proevenverzameling samen te stellen is beschreven in een vaste structuur. Deze structuur is vastgelegd in een uitwisselformat. Het uitwisselformat-database-proevenverzameling_versie_4_2x.xlsx. De geotechnische laboratoria kennen deze database en kunnen deze database vullen met resultaten van grond- en laboratoriumonderzoek. Op deze wijze ontstaat er uniformering op het gebied van data-uitwisseling en –opslag van proefresultaten.

NB. De verantwoordelijkheid voor het gebruik van deze tools ligt bij de gebruiker.

## Inhoudsopgave

- [Stap 1: Opgeven benodigde data en export locatie](#Stap-1:-Opgeven-benodigde-data-en-export-locatie)
- [Stap 2: Importeren en valideren](#Stap-2a:-Importeren-en-valideren-data-(inclusief-toevoegen/herberekenen-analyse-kolommen))
- [Stap 3: Kies materiaalfactor en alpha](#Stap-3:-Instellingen-(materiaalfactor-en-alpha))
- [Stap 4: C-Phi analyse](#Stap-4:-Bepalen-gedraineerde-parameters-triaxiaalproeven-en-DSS-proeven-obv-fit-(C-en-Phi)-of-obv-schematiseringshandleiding-(Phi))
- [Stap 5: SHANSEP](#Stap-5:-Bepalen-ongedraineerde-parameters-triaxiaalproeven-en-DSS-proeven-obv-methode-Shansep-(S,-m-en-POP))
- [Stap 6: SU-Tabel](#Stap-6:-Bepalen-ongedraineerde-parameters-triaxiaalproeven-en-DSS-proeven-obv-Su-tabel-methode)
- [Extra: Figuur genereren](#Extra:-Genereren-van-een-figuur)

## Benodigde installaties

Voor het werken met de PV-tool dient het benodigd package te worden geinstalleerd. Hierin staat de achterliggende code om deze tool werkbaar te maken. 

Ook dienen verschillende python-packages te worden geinstalleerd. Na de installatie dient de kernel handmatig opnieuw opgestart te worden via het menu Kernel -> Restart.

In [1]:
# Importeren van benodigde package
# Na het eerte keer installeren van de benodigde packages de kernel handmatig opnieuw optarten via het menu Kernel → Restart

path_to_wheel = r"C:\Users\deenekat7271\Documents\GitHub\pv-tool\dist\pv_tool-0.3.4-py3-none-any.whl"
!pip install "{path_to_wheel}"

Defaulting to user installation because normal site-packages is not writeable
Processing .\dist\pv_tool-0.3.4-py3-none-any.whl
pv-tool is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.


In [2]:
# all imports
import importlib.util
import ipywidgets as widgets
from IPython.display import display, Markdown
from ipyfilechooser import FileChooser
from pathlib import Path
import os
import plotly.io as pio
pio.renderers.default = "notebook_connected"

from pv_tool.imports.import_data import Dbase
from pv_tool.cphi_analysis.c_phi_analysis import CPhiAnalyse
from pv_tool.cphi_analysis.variables import *

# widgetfuncties nog verder aanvullen!
from pv_tool.utilities.widget_functions_cphi import *
from pv_tool.utilities.widget_functions_shansep import *
from pv_tool.utilities.widget_functions_su import *

## Stap 1: Opgeven benodigde data en export locatie

In [3]:
import_dropdown, import_filechooser, export_dirchooser, export_namebox = setup_interactive_import_export()

Dropdown(description='Template:', index=2, layout=Layout(width='400px'), options=('Proevenverzamelingtool 4.2n…

FileChooser(path='C:\', filename='', title='Selecteer een bestand om te uploaden', show_hidden=False, select_d…

Output()

## Stap 2a: Importeren en valideren data (inclusief toevoegen/herberekenen analyse kolommen)

### Importeren

De importfunctie laadt de data in. Daarnaast worden analyse kolommen (ANA) toegevoegd die benodigd zijn voor de functies in "shansep_analysis" en "sutabel_analysis".

Hiervoor worden verschillende kolommen aangemaakt: ANA_TERREINSPANNING, ANA_TXT_MAX_VERTICALE_CONSOLIDATIE_SPANNING, ANA_DSS_MAX_CONSOLIDATIE_SPANNING, ANA_TXT_CONSOLIDATIE_TYPE_VOORSTEL, ANA_TXT_CONSOLIDATIE_TYPE_HANDMATIG, ANA_TXT_CONSOLIDATIE_TYPE_REKEN, ANA_DSS_CONSOLIDATIE_TYPE_VOORSTEL, ANA_DSS_CONSOLIDATIE_TYPE_HANDMATIG ANA_DSS_CONSOLIDATIE_TYPE_REKEN, ANA_GRENSSPANNING_PROEF, ANA_POP_VELD, ANA_POP_VELD_GEMIDDELD, ANA_GRENSSPANNING_VOORSTEL, ANA_GRENSSPANNING_HANDMATIG ANA_GRENSSPANNING_REKEN, OCR_TXT, OCR_DSS.

Er moet worden vastgesteld of een proef is uitgevoerd op een normaal geconsolideerd monster (NC) ruim boven de grensspanning of overconsolideerd (OC) bij de geschatte dagelijkse effectieve terreinspanning (OCR>1). De tooling doet hier een voorstel voor door de consolidatiespanning (sigma'vc) te vergelijken met de terreinspanning (sigma'vi). Indien deze niet meer dan +30% afwijkt wordt aangenomen dat het een OC-proef betreft en bij grotere afwijkingen een NC-proef. Het is noodzakelijk om dit voorstel te controleren. In de regel is een OC-proef vrijwel altijd dicht bij de terreinspanning gekozen en ligt een NC-proef er ruim boven, maar hier zijn altijd uitzonderingen op mogelijk.

Daarnaast is een schatting van de OCR benodigd. Om deze te berekenen is een schatting van de grensspanning benodigd. Indien er op dezelfde regel een resultaat beschikbaar is uit een samendrukkingsproef of CRS-proef wordt deze gebruikt. Indien deze ontbreekt wordt een voorstel gedaan op basis van de gemiddelde POP (POP = grensspanning - effectieve terreinspanning o.b.v. de monsters waar wel een proef is uitgevoerd ) per boring. Indien er bij de betreffende boring geen grensspanning beschikbaar is wordt geen waarde ingevuld. Het is dan niet mogelijk om geautomatiseerd een OCR te schatten. De data wordt weggeschreven in de velden (ANA_GRENSSPANNING_VOORSTEL). Het is noodzakelijk om de voorgestelde waarden te controleren. Als gebruiker kan je het voorstel overrulen in de kolom (ANA_GRENSSPANNING_HANDMATIG).

NB. Bij gebruik van de bestaande Excel versie van de proevenverzamelingtool zijn de cellen ANA_GRENSSPANNING_HANDMATIG veelal reeds ingevuld. In dat geval worden deze waarden overgenomen.

### Valideren


De validatie bestaat uit de volgende controles: voor de verschillende typen proeven – classificatie, CRS-proef, samendrukkingsproef, DSS-proef en triaxiaalproef – wordt per regel gecontroleerd of er een proef is uitgevoerd (data ingevuld) en zo ja, of de velden die in de PV-tool benodigd zijn volledig en correct zijn ingevuld.

Het resultaat van de validatie wordt opgeslagen in een twee Excel-bestanden Validation_log_##critical_errors.xlsx en Validation_log##_warnings.xlsx, waarin per gevalideerde kolom en regel het resultaat wordt weergegeven. Ir is per kolom aangegeven hoeveel fouten zijn aangetroffen. Er is onderscheid gemaakt tussen 'critical errors' en 'warnings'. De critical errors leiden tot fouten wanneer deze data wordt gebruikt bij het bepalen van de parameters. Deze dienen te worden gecorrigeerd of aangevuld. De warnings betreffen aanbevelingen. De resultaten worden weggeschreven in dezelfde directory als de importmap.

Indien de validatie goed is doorlopen worden proevenverzamelingtool (vanaf versie 4.2n of hoger) of Uitwisselformat-database-proevenverzameling_versie_4_2x.xlsx weggeschreven naar het Template_PVtool5_0.xlsx. Indien reeds gebruik is gemaakt van het nieuwe template worden de analyse kolommen opnieuw berekend. Dit is noodzakelijk indien er data gewijzigd is.

Indien er reeds een gevalideerde dataset beschikbaar is conform het Template_PVtool5_0 laadt de data dan in via stap 2b. Dit kost aanzienlijk minder rekentijd bij een grote dataset.

In [ ]:
dbase = Dbase()
handle_import_export(
    dbase,
    import_dropdown,
    import_filechooser,
    export_dirchooser,
    process_import_and_validate
)

## Stap 2b: Als errors zijn opgelost exporteer naar Template

In [ ]:
# Export dbase-template
if import_dropdown.value == "Proevenverzamelingtool 5.0":
    print('Export gelijk aan import')
else:
    process_export(
        dbase,
        export_dir=Path(export_dirchooser.selected_path),
        filename=export_namebox.value if export_namebox.value else None
    )

## Stap 2b: Importeren data zonder validatie of herberekening analysekolommen

### LET OP! ALLEEN MOGELIJK VOOR TEMPLATE PROEVENVERZAMELINGTOOL 5.0

De importfunctie laadt de data conform het Template_PVtool5_0. Deze functie mag alleen gebruikt worden indien er reeds een gevalideerde dataset beschikbaar is conform het Template_PVtool5_0. Na het aanbrengen van wijzigingen in de data altijd stap 2a toepassen voor het importeren. Dit is noodzakelijk omdat de analsyekolommen benodigd voor onderstaande functies opnieuw berekend moeten worden bij aanpassingen aan de data.

In [4]:
dbase = Dbase()
handle_import_only(
    dbase,
    import_dropdown,
    import_filechooser,
    process_import_only
)

**Template-code:** Dbase

**Data geïmporteerd uit:** C:\Users\deenekat7271\Documents\GitHub\pv-tool\Template_PVtool5_0.xlsx

## Stap 3: Instellingen (materiaalfactor en alpha)

Kies materiaalfactor en alpha op basis van de type verzameling. 

In [5]:
# Stel materiaalfactor en alpha in
alpha_widget, partphi_widget, partcoh_widget = toon_grid_settings()

**Pas alpha aan (lokaal = 1,0, regionaal = 0,75):**

**Pas materiaalfactoren aan:**

## Stap 4: Bepalen gedraineerde parameters triaxiaalproeven en DSS-proeven obv fit (C en Phi) of obv schematiseringshandleiding (Phi)

[Terug naar boven](#Inhoudsopgave)

In deze stap worden op basis van de data vastgelegd in het template PV-tool 5.0. gedraineerde parameters bepaald. Maak een keuze uit de verzameling waarop de statistische analyse moet plaatsvinden. Als er nog geen aparte verzamelingen onderscheiden zijn in het veld (PV_NAAM) worden alle triaxaalproeven in één verzameling opgenomen.

Als gebruiker kunnen handmatig verzamelingen worden opgegeven in de Excel template in het veld PV_NAAM of er kan gebruik worden gemaakt van de functie: Definiëren en aanpassen van verzamelingen (stap #). Met behulp van de functie ## kunnen in de grafiek met behulp van de selectietool (should have) ook één of meer proefresultaten worden geselecteerd en worden toegekend aan een andere groep.

Geef het te hanteren rekpercentage op waarbij de s' en t zijn bepaald: 2%, 5%, 15%, eindrek of bij pieksterkte. Indien er eerder parameters zijn vastgesteld worden de gemaakte keuzes voor het % en de raaklijnen ingeladen.

Ook kunnen meerdere verzamelingen getoond worden. NB. Deze extra verzamelingen worden niet meegenomen in de statistische analyse.

In [ ]:
(dropdown_type_proef, dropdown_verzameling, dropdown_rekpercentage_txt, 
 dropdown_rekpercentage_dss, container_rekpercentage, output_rekpercentage, 
 multi_select_verzameling, gekozen_rekpercentage) = dropdown_widgets(dbase)

In [ ]:
# run analyse
analyse, coh_gem, phi_kar, coh_kar = voer_cphi_analyse_uit(
    dbase=dbase,
    import_dropdown=import_dropdown,
    import_filechooser=import_filechooser,
    dropdown_verzameling=dropdown_verzameling,
    dropdown_type_proef=dropdown_type_proef,
    dropdown_rekpercentage_txt=dropdown_rekpercentage_txt,
    dropdown_rekpercentage_dss=dropdown_rekpercentage_dss,
    export_dir_widget=export_dirchooser,
    export_name_widget=export_namebox,
    gekozen_rekpercentage=gekozen_rekpercentage,
    toon_cphi_tabel=toon_cphi_tabel,
    partphi_widget=partphi_widget,
    partcoh_widget=partcoh_widget,
    alpha_widget=alpha_widget
)

In [ ]:
# Voer C-Phi analyse uit met invoer en toon grafieken
analyse, output_df = show_cphi_analysis(
    dbase,
    dropdown_verzameling,
    dropdown_type_proef,
    dropdown_rekpercentage_txt,
    dropdown_rekpercentage_dss,
    coh_gem,
    phi_kar,
    coh_kar,
    multi_select_verzameling,
    alpha_widget,
    partphi_widget,
    partcoh_widget
)

In [ ]:
# Resultaten opslaan in export-bestand
_ = export_results(
    analyse,
    dropdown_verzameling,
    dropdown_type_proef,
    import_dropdown,
    import_filechooser,
    export_dir_widget=export_dirchooser,
    export_name_widget=export_namebox
)

## Stap 5: Bepalen ongedraineerde parameters triaxiaalproeven en DSS-proeven obv methode Shansep (S, m en POP)

[Terug naar boven](#Inhoudsopgave)

In deze stap worden op basis van de data vastgelegd in het template PV-tool 5.0. de gedraineerde parameters bepaald volgens de methode vastgelegd in de schematiseringshandleiding macrostabiliteit. Maak een keuze uit de verzameling waarop de statistische analyse moet plaatsvinden. Als er nog geen aparte verzamelingen onderscheiden zijn in het veld (PV_NAAM) worden alle triaxaalproeven in één verzameling opgenomen.

Als gebruiker kunnen handmatig verzamelingen worden opgegeven in de Excel template in het veld PV_NAAM of er kan gebruik worden gemaakt van de functie: Definiëren en aanpassen van verzamelingen (stap #). Met behulp van de functie ## kunnen in de grafiek met behulp van de selectietool (should have) ook één of meer proefresultaten worden geselecteerd en worden toegekend aan een andere groep.

Geef het te hanteren rekpercentage op waarbij de Su is bepaald: 2%, 5%, 15%, eindrek of bij pieksterkte. Indien er eerder parameters zijn vastgesteld worden de gemaakte keuzes voor het % en de raaklijnen ingeladen.

Ook kunnen meerdere verzamelingen getoond worden. NB. Deze extra verzamelingen worden niet meegenomen in de statistische analyse.

In [ ]:
(dropdown_type_proef_shansep, dropdown_verzameling_shansep, dropdown_rekpercentage_txt_shansep, 
 dropdown_rekpercentage_dss_shansep, container_rekpercentage_shansep, output_rekpercentage_shansep,
 multi_select_verzameling_shansep, gekozen_rekpercentage_shansep) = dropdown_widgets_shansep(dbase)

In [ ]:
# Run analyse en show grid
analyse, df_gem, df_kar, widgets_gem, widgets_kar = run_shansep_analysis(
        dbase,
        dropdown_verzameling_shansep,
        alpha_widget,
        import_dropdown,
        import_filechooser,
        dropdown_rekpercentage_txt_shansep,
        dropdown_rekpercentage_dss_shansep,
        dropdown_type_proef_shansep,
        export_dir_widget=None,
        export_name_widget=None
)

In [ ]:
# Voer SHANSEP-analyse uit met invoer en toon grafieken
analyse, output_df = show_shansep_analysis(
    dbase,
    widgets_kar,
    widgets_gem,
    dropdown_verzameling_shansep,
    dropdown_type_proef_shansep,
    dropdown_rekpercentage_txt_shansep,
    dropdown_rekpercentage_dss_shansep,
    multi_select_verzameling_shansep,
    alpha_widget
)

In [ ]:
# Exports
_ = export_shansep_results(
    analyse,
    dropdown_verzameling_shansep,
    dropdown_type_proef_shansep,
    import_dropdown,
    import_filechooser,
    export_dir_widget=export_dirchooser,
    export_name_widget=export_namebox)

## Stap 6: Bepalen ongedraineerde parameters triaxiaalproeven en DSS-proeven obv Su tabel methode

[Terug naar boven](#Inhoudsopgave)

In deze stap worden op basis van de data vastgelegd in het template PV-tool 5.0. de gedraineerde parameters bepaald volgens de methode vastgelegd in de schematiseringshandleiding macrostabiliteit. Maak een keuze uit de verzameling waarop de statistische analyse moet plaatsvinden. Als er nog geen aparte verzamelingen onderscheiden zijn in het veld (PV_NAAM) worden alle triaxaalproeven in één verzameling opgenomen.

Als gebruiker kunnen handmatig verzamelingen worden opgegeven in de Excel template in het veld PV_NAAM of er kan gebruik worden gemaakt van de functie: Definiëren en aanpassen van verzamelingen (stap #). Met behulp van de functie ## kunnen in de grafiek met behulp van de selectietool (should have) ook één of meer proefresultaten worden geselecteerd en worden toegekend aan een andere groep.

Geef het te hanteren rekpercentage op waarbij de Su is bepaald: 2%, 5%, 15%, eindrek of bij pieksterkte. Indien er eerder parameters zijn vastgesteld worden de gemaakte keuzes voor het % en de raaklijnen ingeladen.

Ook kunnen meerdere verzamelingen getoond worden. NB. Deze extra verzamelingen worden niet meegenomen in de statistische analyse.

In [6]:
(dropdown_type_proef_su, dropdown_verzameling_su, dropdown_rekpercentage_txt_su,
 dropdown_rekpercentage_dss_su, container_rekpercentage_su, output_rekpercentage_su,
 multi_select_verzameling_su, gekozen_rekpercentage_su) = dropdown_widgets_su(dbase)

**Kies type proef:**

Dropdown(description='Type proef:', layout=Layout(width='400px'), options=('TXT_su_tabel', 'DSS_su_tabel'), va…

**Kies verzameling voor statistische analyse:**

Dropdown(description='Verzameling:', layout=Layout(width='400px'), options=('geen', 'TXT_testset_klei', 'TXT_W…

**Kies rekpercentage Su:**

Output()

Output()

**Kies één of meerdere verzamelingen om naast de gekozen verzameling voor de statistische analyse te tonen:**

SelectMultiple(description='Vergelijk met:', index=(0,), layout=Layout(height='150px', width='400px'), options…

In [7]:
# run analyse en show grid
analyse, handmatige_widgets = run_su_analysis(
    dbase,
    dropdown_verzameling_su,
    alpha_widget,
    import_dropdown,
    import_filechooser,
    dropdown_rekpercentage_txt_su,
    dropdown_rekpercentage_dss_su,
    dropdown_type_proef_su,
    export_dir_widget=None,
    export_name_widget=None)

**Opgegeven karakteristieke waarden (fit):**

GridspecLayout(children=(Label(value='Parameters', layout=Layout(grid_area='widget001', width='200px')), HTML(…

In [9]:
# show results
analyse, output_df = show_su_analysis(dbase,
        dropdown_verzameling_su,
        dropdown_type_proef_su,
        dropdown_rekpercentage_txt_su,
        dropdown_rekpercentage_dss_su,
        handmatige_widgets[0].value,
        handmatige_widgets[1].value,
        handmatige_widgets[2].value,
        multi_select_verzameling_su,
        alpha_widget,
        import_dropdown,
        import_filechooser                      
)

                        svgm [kPa]     m [-]  vc_fit [-]
Verwachtingswaarde        1.875285  0.345547        0.01
Karakteristieke waarde    1.733253       0.3        0.01
Standaarddeviatie         0.143834       [-]        0.01


In [10]:
# Export results
_ = export_su_results(
    analyse,
    dropdown_verzameling_su,
    dropdown_type_proef_su,
    import_dropdown,
    import_filechooser,
    export_dir_widget=export_dirchooser,
    export_name_widget=export_namebox
)

Resultaat toegevoegd aan template in tabblad 'Resultaten SU-tabel-m'.
Figuren opgeslagen als HTML in: C:\Users\deenekat7271\Documents\GitHub\pv-tool
Sutabel PDF export voltooid: C:\Users\deenekat7271\Documents\GitHub\pv-tool/sutabel_pdf_export_TXT_SAFE_klei_licht_16_175_TXT_su_tabel_eindsterkte.pdf


## Extra: Genereren van een figuur

[Terug naar boven](#Inhoudsopgave)

In [ ]:
import plotly.express as px

import_data = dbase.dbase_df

fig = px.scatter(
    import_data,
    x='CLAS_MONSTERNIVEAU', 
    y='CLAS_VOLUMEGEWICHT_DRG',
    color='PV_NAAM',                     # kleur op basis van PV_NAAM
    title='Droog volumegewicht tegen het monsterniveau',
    labels={
        'CLAS_MONSTERNIVEAU': 'Monsterniveau',
        'CLAS_VOLUMEGEWICHT_DRG': 'Droog volumegewicht',
        'PV_NAAM': 'PV naam'
    },
    template='plotly_white',
    opacity=0.8,
    hover_data=['PV_NAAM']               # voeg extra hover-info toe indien gewenst
)

fig.update_traces(marker=dict(size=7))   # pas marker-grootte aan
fig.show()

In [ ]:
if import_dropdown.value == 'Proevenverzamelintool 5.0':
    export_dir = str(import_path.parent)
else:
    try:
        selected = getattr(export_dirchooser, "selected_path", None) or getattr(export_dirchooser, "selected", None)
        if selected:
            export_dir = str(selected)
        else:
            export_dir = str(getattr(export_dirchooser, "current_path", export_dirchooser.path))
    except Exception as e:
        raise RuntimeError("Kon exportmap niet ophalen uit export_dirchooser: {}".format(e))

naam_plaatje = 'extra_plaatje'

export_path = Path(export_dir)
output_html = export_path / f"{naam_plaatje}.html"

fig.write_html(str(output_html))
print(f"HTML figuur opgeslagen in: {output_html}")